In [ ]:
# pip install h3

In [ ]:
# pip install contextily

In [ ]:
import geopandas as gpd
import numpy as np
import h3
import networkx as nx
from shapely.geometry import Point, LineString
import pandas as pd
import matplotlib.pyplot as plt
import contextily as ctx

In [1]:
import geopandas as gpd
import networkx as nx
from shapely.geometry import LineString, Point
import h3

def create_network_from_h3(h3_shp_path, output_line_shp_path):
    h3_gdf = gpd.read_file(h3_shp_path)

    # Create a lookup for centroids by h3 index
    centroids = {}
    for _, row in h3_gdf.iterrows():
        h3_index = row['h3_index']
        centroids[h3_index] = row.geometry.centroid

    # Create graph
    G = nx.Graph()

    for h3_index, centroid in centroids.items():
        G.add_node(h3_index, pos=(centroid.x, centroid.y))

    # Add edges between neighbors
    for h3_index in centroids:
        neighbors = h3.k_ring(h3_index, 1) - {h3_index}
        for neighbor in neighbors:
            if neighbor in centroids and not G.has_edge(h3_index, neighbor):
                point1 = centroids[h3_index]
                point2 = centroids[neighbor]
                distance = point1.distance(point2)
                G.add_edge(h3_index, neighbor, weight=distance)

    # Convert edges to GeoDataFrame
    lines = []
    for node1, node2, data in G.edges(data=True):
        pos1 = G.nodes[node1]['pos']
        pos2 = G.nodes[node2]['pos']
        line = LineString([pos1, pos2])
        lines.append({
            'geometry': line,
            'from_node': node1,
            'to_node': node2,
            'length': data['weight']
        })

    line_gdf = gpd.GeoDataFrame(lines, crs=h3_gdf.crs)
    line_gdf.to_file(output_line_shp_path)

    return G, line_gdf


ModuleNotFoundError: No module named 'h3'

In [ ]:
def all_or_nothing_assignment(G, od_matrix, h3_indices):
    """
    Performs all-or-nothing traffic assignment on the network.

    Args:
        G: NetworkX graph
        od_matrix: numpy matrix of origin-destination flows
        h3_indices: list of H3 indices in the same order as the OD matrix
    """
    # Create a dictionary to store flows on edges
    edge_flows = {edge: 0 for edge in G.edges()}

    # For each OD pair
    for i, origin in enumerate(h3_indices):
        for j, destination in enumerate(h3_indices):
            if i != j and od_matrix[i, j] > 0:
                # Find shortest path
                try:
                    path = nx.shortest_path(G, origin, destination, weight='weight')

                    # Add flow to each edge in the path
                    for k in range(len(path) - 1):
                        edge = (path[k], path[k + 1])
                        edge_flows[edge] += od_matrix[i, j]
                except nx.NetworkXNoPath:
                    print(f"No path found between {origin} and {destination}")

    return edge_flows


In [ ]:
def create_flow_map(line_gdf, output_png_path):
    """
    Creates a map visualization of the network with flows represented by line widths.

    Args:
        line_gdf: GeoDataFrame containing the network lines and flows
        output_png_path: Path to save the output PNG file
    """
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(15, 15))

    # Normalize flows for line widths
    max_flow = line_gdf['flow'].max()
    line_gdf['line_width'] = (line_gdf['flow'] / max_flow) * 5  # Scale factor of 5 for visibility

    # Plot the network
    line_gdf.plot(ax=ax,
                 color='red',
                 linewidth=line_gdf['line_width'],
                 alpha=0.6)

    # Add basemap
    ctx.add_basemap(ax,
                    crs=line_gdf.crs.to_string(),
                    source=ctx.providers.CartoDB.Positron)

    # Remove axes
    ax.set_axis_off()

    # Add title
    plt.title('Traffic Flow Network', pad=20, fontsize=16)

    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap='YlOrRd',
                              norm=plt.Normalize(vmin=0, vmax=max_flow))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', pad=0.01)
    cbar.set_label('Traffic Flow')

    # Save the map
    plt.savefig(output_png_path,
                dpi=300,
                bbox_inches='tight',
                facecolor='white')
    plt.close()


In [ ]:
def main():
    # Example usage
    h3_shp_path = "input_h3.shp"  # Replace with your input H3 shapefile path
    output_line_shp_path = "network_lines.shp"
    output_map_path = "flow_map.png"

    # Create network
    G, line_gdf = create_network_from_h3(h3_shp_path, output_line_shp_path)

    # Example OD matrix (replace with your actual matrix)
    h3_indices = list(G.nodes())
    n = len(h3_indices)
    od_matrix = np.random.rand(n, n)  # Replace with your actual OD matrix

    # Perform traffic assignment
    edge_flows = all_or_nothing_assignment(G, od_matrix, h3_indices)

    # Add flows to the line GeoDataFrame
    line_gdf['flow'] = line_gdf.apply(
        lambda row: edge_flows.get((row['from_node'], row['to_node']), 0),
        axis=1
    )

    # Save the results
    line_gdf.to_file("assigned_network.shp")

    # Create and save the flow map
    create_flow_map(line_gdf, output_map_path)


In [ ]:

if __name__ == "__main__":
    main()